# The Receipts Notebook

*The Backtest Graveyard. Every aired episode in one file. Updated 2026-11-28.*

Every episode ends with a verdict. This is where the verdicts come from.

One section per episode: what got tested, where the data came from, how to rerun it, and what the test said. Every figure is quoted word for word from that episode's write-up in the public receipts repo. The code cell under each section goes and checks. Run it and it prints the line each figure came from. If one ever comes back MISSING, the write-up changed and I owe you a correction.

Not financial advice, and I'm not licensed to give it. A backtest is a simulation of the past. It isn't a prediction, and nothing here tells you what to buy.

How to use it. Read it as it is, or [open it in Colab](https://colab.research.google.com/github/backtest-graveyard/backtest-graveyard.github.io/blob/main/receipts/receipts_notebook.ipynb) and pick Runtime, then Run all. No account, no keys, nothing to install. The checks read the write-ups straight from GitHub. Rerunning a backtest itself is different: use the command under each episode, and some of them need your own market-data keys. Each section says which. All the code and charts live in [the receipts repo](https://github.com/backtest-graveyard/backtest-graveyard).

## The scoreboard

| Ep | Test | Stamp |
|---|---|---|
| 1 | [The Overnight Effect](#ep01) | DEAD. |
| 2 | [Gap-and-Go](#ep02) | DEAD. |
| 3 | [The Kelly Criterion](#ep03) | WORSE. |
| 4 | [The 51% → $100 Billion Math](#ep04) | NOT AN EDGE. |
| 5 | [GEX and Dealer Hedging](#ep05) | HALF-REAL. |
| 6 | [Market-Neutral Stat-Arb](#ep06) | NOT AT RETAIL. |
| 7 | [Kronos, the "GPT for candlesticks" AI](#ep07) | BEATEN BY A FORMULA. |
| 8 | [The Viral "VIX Cheat Sheet"](#ep08) | HALF-TRUE. |
| 9 | [Buy QQQM Below Its 200-Day, Trail Out With a 2% Stop](#ep09) | BEATEN BY DOING NOTHING. |
| 10 | [MACD + the "Profitability Multiplier" Volume Filter](#ep10) | A THROTTLE, NOT AN EDGE. |
| 11 | [Dollar-Cost Averaging vs. Lump-Sum](#ep11) | SAFER, NOT RICHER. |
| 12 | [How Much Backtesting Is Enough?](#ep12) | NOT THE QUESTION. |
| 13 | [TQQQ "Just Buy 3x And Hold"](#ep13) | WORSE RIDE. |
| 14 | [RSI-2 "It Calls The Bottom"](#ep14) | NOT AN EDGE. |
| 15 | [The Momentum Factor, Retail Edition](#ep15) | JUST BETA. |
| 16 | [The Golden Cross (50/200)](#ep16) | JUST A SEATBELT. |
| 17 | [HedgeFundie's Excellent Adventure (HFEA)](#ep17) | THE HEDGE BROKE. |
| 18 | [Covered-Call Income ETFs](#ep18) | YOUR OWN MONEY. |
| 19 | [Sell in May and Go Away](#ep19) | STAY IN MAY. |
| 20 | [4-Hour vs Daily, Crypto: The Re-Test](#ep20) | REVERSED. |
| 21 | [Buffett's Rules, Tested](#ep21) | NOT VALIDATED. |

In [1]:
# Standard library only. Runs in Colab, Jupyter, or plain Python.
import os, urllib.request

RAW = "https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/"
LOCAL = os.path.expanduser("~/backtest-graveyard")  # used instead of GitHub if you cloned the repo
_readmes = {}
TALLY = []

def readme(folder):
    """The episode's write-up, exactly as committed in the receipts repo."""
    if folder not in _readmes:
        path = os.path.join(LOCAL, folder, "README.md")
        if os.path.exists(path):
            _readmes[folder] = open(path, encoding="utf-8").read()
        else:
            with urllib.request.urlopen(RAW + folder + "/README.md") as r:
                _readmes[folder] = r.read().decode("utf-8")
    return _readmes[folder]

def receipts(folder, quotes):
    """Find each quoted figure in the episode's README and print the line it sits on."""
    lines = [l.replace("**", "").replace("*", "").replace("`", "") for l in readme(folder).splitlines()]
    for q in quotes:
        line = next((i + 1 for i, l in enumerate(lines) if q in l), None)
        TALLY.append(line is not None)
        print(("found   " if line else "MISSING ") + f"{folder}/README.md line {line or '-'}")
        print("        " + q)

<a id="ep01"></a>
## Ep 1. The Overnight Effect

**Stamp: DEAD.**

**The test.** A viral post claimed that buying Micron (MU) at every close and selling at every open made a fortune. The test checked whether the overnight/intraday split is real, then whether it can be harvested every night or only across earnings nights. The kill tests dropped the best nights and swapped the hand-picked names for a point-in-time universe (only names you could have listed back then).

**The data.** SIP daily bars from 2016-01-04 to 2026-08-21 via Alpaca's API. Earnings dates and times come from SEC EDGAR 8-K filings with Item 2.02.

**Rerun it.** Three scripts are committed: overnight_earnings_backtest.py, overnight_earnings_ptuniverse.py and make_charts.py. They need Python 3.11+, matplotlib, your own Alpaca market-data keys, and your own contact info in the SEC User-Agent header.

```bash
python3 overnight_earnings_backtest.py
```

**The verdict.** Dead. The overnight anomaly is real, but no tradable version of it survives honest testing. Trading it every night dies on costs, earnings nights collapse without the five best, and the point-in-time universe has nothing.

**The numbers, quoted from the write-up:**

> Verdict: DEAD. The viral overnight-return anomaly is real, but no tradable version of it survives honest testing.
>
> At 5 bps per round trip, 2,674 days of daily trading multiplies terminal equity by ~0.26; at 10 bps, ~0.07.
>
> | Drop 5 best nights | +0.7%/yr | +4.1%/yr |
>
> Removing five nights out of 42 collapses both books: the tails ARE the strategy.
>
> | TOP-20 (primary) | +2.0% | 0.07 |
>
> Monotone: the more honest the universe, the worse the book.

![Ep 1 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep01-overnight-effect/charts/chart3_kill_ladder.png)

Receipts: [ep01-overnight-effect/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep01-overnight-effect) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=83HvnC28VKs)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [2]:
receipts("ep01-overnight-effect", [
    "Verdict: DEAD. The viral overnight-return anomaly is real, but no tradable version of it survives honest testing.",
    "At 5 bps per round trip, 2,674 days of daily trading multiplies terminal equity by ~0.26; at 10 bps, ~0.07.",
    "| Drop 5 best nights | +0.7%/yr | +4.1%/yr |",
    "Removing five nights out of 42 collapses both books: the tails ARE the strategy.",
    "| TOP-20 (primary) | +2.0% | 0.07 |",
    "Monotone: the more honest the universe, the worse the book.",
])

found   ep01-overnight-effect/README.md line 4
        Verdict: DEAD. The viral overnight-return anomaly is real, but no tradable version of it survives honest testing.
found   ep01-overnight-effect/README.md line 30
        At 5 bps per round trip, 2,674 days of daily trading multiplies terminal equity by ~0.26; at 10 bps, ~0.07.
found   ep01-overnight-effect/README.md line 46
        | Drop 5 best nights | +0.7%/yr | +4.1%/yr |
found   ep01-overnight-effect/README.md line 48
        Removing five nights out of 42 collapses both books: the tails ARE the strategy.
found   ep01-overnight-effect/README.md line 66
        | TOP-20 (primary) | +2.0% | 0.07 |
found   ep01-overnight-effect/README.md line 70
        Monotone: the more honest the universe, the worse the book.


<a id="ep02"></a>
## Ep 2. Gap-and-Go

**Stamp: DEAD.**

**The test.** A gap-and-go momentum bot was backtested across every US stock: gap ≥ 10%, price $2–$20, float < 50M, RVOL ≥ 5×, entry on a breakout on 1-min bars. The kill test strips the best trades by R-multiple (profit in units of the risk taken). Each discretionary read was also turned into a rule and scored against real trade outcomes.

**The data.** Intraday bars for every US stock, 2025-09-11 to 2026-04-17, giving 537 trades. The reads analysis used additional professional-grade market data and EDGAR offering history.

**Rerun it.** reproduce.py recomputes every kill-test number from the committed trades.csv. It needs only Python 3.8+, plus matplotlib for --charts, and no credentials. The PAVS chart, the ROI sweep and the reads analysis aren't re-runnable from this folder.

```bash
python3 reproduce.py
```

**The verdict.** Teachable, not deployable. The pattern is real, but the profit is a handful of unpredictable monster mornings a bot can't capture on retail data and fills. The one mechanizable edge that survives is avoiding serial diluters.

**The numbers, quoted from the write-up:**

> Verdict: teachable, not deployable.
>
> Headline (as-shipped, clean fills): +$12,505 over the window, +36.6R, 44% win rate.
>
> | minus top 5 | −14.7R — goes negative |
>
> The strategy doesn't have 500 small edges; it has 5 lottery tickets and 532 scratch-offs.
>
> The honest middle case (~+10%) merely matches the index
>
> Skipping the serial diluters is worth ~+0.15R per trade.

![Ep 2 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep02-gap-and-go/charts/chart3_strip_ladder.png)

Receipts: [ep02-gap-and-go/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep02-gap-and-go) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=RF2WZ8EH1vE)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [3]:
receipts("ep02-gap-and-go", [
    "Verdict: teachable, not deployable.",
    "Headline (as-shipped, clean fills): +$12,505 over the window, +36.6R, 44% win rate.",
    "| minus top 5 | −14.7R — goes negative |",
    "The strategy doesn't have 500 small edges; it has 5 lottery tickets and 532 scratch-offs.",
    "The honest middle case (~+10%) merely matches the index",
    "Skipping the serial diluters is worth ~+0.15R per trade.",
])

found   ep02-gap-and-go/README.md line 4
        Verdict: teachable, not deployable.
found   ep02-gap-and-go/README.md line 24
        Headline (as-shipped, clean fills): +$12,505 over the window, +36.6R, 44% win rate.
found   ep02-gap-and-go/README.md line 32
        | minus top 5 | −14.7R — goes negative |
found   ep02-gap-and-go/README.md line 35
        The strategy doesn't have 500 small edges; it has 5 lottery tickets and 532 scratch-offs.
found   ep02-gap-and-go/README.md line 62
        The honest middle case (~+10%) merely matches the index
found   ep02-gap-and-go/README.md line 84
        Skipping the serial diluters is worth ~+0.15R per trade.


<a id="ep03"></a>
## Ep 3. The Kelly Criterion

**Stamp: WORSE.**

**The test.** Does handing a working bot's position sizing to the Kelly criterion (a formula for the "optimal" bet fraction) make it better? The bot's signal stayed untouched. Constant-risk sizing was raced against quarter, half and full Kelly, with every variant matched to the same average gross exposure so none could win just by betting bigger.

**The data.** 5 years of history from a private, live paper-traded bot, with costs of 15 bps entry and 8 bps exit. The README doesn't name the price data source.

**Rerun it.** No code is committed. The bot's signal and code are private, so the results aren't independently re-runnable. Every figure is in kelly_sizing_results.csv, and the method is described for running the same test on your own strategy.

**The verdict.** Kelly was a downgrade on a bot that already works, and full strength was the worst. MAR (yearly return divided by the worst drop) fell from 2.05 at constant risk to 1.42 at full Kelly. Keep boring, constant-risk sizing.

**The numbers, quoted from the write-up:**

> Verdict: on a bot that already works, swapping constant-risk sizing for the Kelly criterion was a downgrade. At full strength it was the worst. Keep boring, constant-risk sizing.
>
> | Constant-risk (vol-target) | 0.35 | +51.2% | 2.05 | 1.96 | −6.2% |
>
> | Kelly ×1.00 (full) | 0.58 | +47.6% | 1.42 | 2.05 | −8.5% |
>
> More Kelly → worse return-to-drawdown, and a deeper hole.
>
> Unmatched, on a single representative asset, full Kelly returned +416% vs +196% for constant-risk sizing over 5 years — a headline "double."

![Ep 3 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep03-kelly-sizing/charts/chart4_mar_ladder.png)

Receipts: [ep03-kelly-sizing/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep03-kelly-sizing) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=_h7k9CxDUGg)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [4]:
receipts("ep03-kelly-sizing", [
    "Verdict: on a bot that already works, swapping constant-risk sizing for the Kelly criterion was a downgrade. At full strength it was the worst. Keep boring, constant-risk sizing.",
    "| Constant-risk (vol-target) | 0.35 | +51.2% | 2.05 | 1.96 | −6.2% |",
    "| Kelly ×1.00 (full) | 0.58 | +47.6% | 1.42 | 2.05 | −8.5% |",
    "More Kelly → worse return-to-drawdown, and a deeper hole.",
    "Unmatched, on a single representative asset, full Kelly returned +416% vs +196% for constant-risk sizing over 5 years — a headline \"double.\"",
])

found   ep03-kelly-sizing/README.md line 3
        Verdict: on a bot that already works, swapping constant-risk sizing for the Kelly criterion was a downgrade. At full strength it was the worst. Keep boring, constant-risk sizing.
found   ep03-kelly-sizing/README.md line 28
        | Constant-risk (vol-target) | 0.35 | +51.2% | 2.05 | 1.96 | −6.2% |
found   ep03-kelly-sizing/README.md line 31
        | Kelly ×1.00 (full) | 0.58 | +47.6% | 1.42 | 2.05 | −8.5% |
found   ep03-kelly-sizing/README.md line 35
        More Kelly → worse return-to-drawdown, and a deeper hole.
found   ep03-kelly-sizing/README.md line 39
        Unmatched, on a single representative asset, full Kelly returned +416% vs +196% for constant-risk sizing over 5 years — a headline "double."


<a id="ep04"></a>
## Ep 4. The 51% → $100 Billion Math

**Stamp: NOT AN EDGE.**

**The test.** A viral post used Renaissance's Medallion Fund to argue that a 51% win rate plus Expected Value, Kelly sizing and the Law of Large Numbers turns into $100 billion. It's a math teardown, not a new backtest. It checks each formula and the post's capacity and cost claims, and reuses the gap-trader numbers from Episode 2.

**The data.** No new market data. The Medallion capacity and cost facts are public, and the gap-trader concentration numbers are the ones published with Episode 2.

**Rerun it.** No code is committed. It's a math teardown with two charts. The portable test is to delete your five best trades and re-run your own backtest.

**The verdict.** Every number in the viral post is real, and the system it describes still has zero edge. Expected Value, Kelly and the Law of Large Numbers size and count your bets. They don't make the money.

**The numbers, quoted from the write-up:**

> Verdict: every number in the viral post is real, and the "system" it describes still has zero edge.
>
> At p = 50.75% with symmetric payoffs the gross edge is ≈ 1.5% of stake per trade.
>
> Add all three together and they create exactly zero edge.
>
> Medallion caps outside AUM at ~$10B and forcibly returns capital and profits to employees
>
> Its effective N was about 3, not 537.

![Ep 4 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep04-rentec-51pct/charts/chart5_capacity.png)

Receipts: [ep04-rentec-51pct/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep04-rentec-51pct) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=9OQisqho4xE)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [5]:
receipts("ep04-rentec-51pct", [
    "Verdict: every number in the viral post is real, and the \"system\" it describes still has zero edge.",
    "At p = 50.75% with symmetric payoffs the gross edge is ≈ 1.5% of stake per trade.",
    "Add all three together and they create exactly zero edge.",
    "Medallion caps outside AUM at ~$10B and forcibly returns capital and profits to employees",
    "Its effective N was about 3, not 537.",
])

found   ep04-rentec-51pct/README.md line 3
        Verdict: every number in the viral post is real, and the "system" it describes still has zero edge.
found   ep04-rentec-51pct/README.md line 15
        At p = 50.75% with symmetric payoffs the gross edge is ≈ 1.5% of stake per trade.
found   ep04-rentec-51pct/README.md line 19
        Add all three together and they create exactly zero edge.
found   ep04-rentec-51pct/README.md line 27
        Medallion caps outside AUM at ~$10B and forcibly returns capital and profits to employees
found   ep04-rentec-51pct/README.md line 43
        Its effective N was about 3, not 537.


<a id="ep05"></a>
## Ep 5. GEX and Dealer Hedging

**Stamp: HALF-REAL.**

**The test.** A viral thread claimed GEX (gamma exposure, an estimate of how options dealers are forced to hedge) pulls price to one level 73% of the time in the first 45 minutes, and that it turned $4,200 into $19,800 in six months. It's a mechanism and failure-mode teardown, not a fresh backtest. It separates the documented volatility-regime effect from the price-magnet and fortune claims.

**The data.** The thread's numbers come from about 60 days of data, and it published no code or trade log. The teardown leans on the SqueezeMetrics 2018 framework and follow-up research, and the charts are illustrative of direction.

**Rerun it.** No code is committed. The write-up and two illustrative charts are here. The GEX estimator needs a live options-chain feed that isn't redistributed.

**The verdict.** Half-real. Dealer gamma does shape whether days are quiet or wild, and that's a documented edge. The 73% price magnet is overstated and broken by same-day options, and the $4,200 → $19,800 is leverage plus survivorship.

**The numbers, quoted from the write-up:**

> Verdict: half-real. Dealer-gamma genuinely shapes the volatility regime (quiet vs. wild) — that read is a real, documented edge.
>
> Mechanism real, fortune a screenshot.
>
> And all the numbers come from ~60 days of data — about one OpEx cycle.
>
> same-day (0DTE) options are now >50% of SPX options volume
>
> plausibly earns Sharpe ~0.7–1.3 (likely 0.5–0.9 net for a retail operator)
>
> Not +371%/6mo.

![Ep 5 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep05-gex-dealer-hedging/charts/chart_regime.png)

Receipts: [ep05-gex-dealer-hedging/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep05-gex-dealer-hedging) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=OXazkvv67Qc)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [6]:
receipts("ep05-gex-dealer-hedging", [
    "Verdict: half-real. Dealer-gamma genuinely shapes the volatility regime (quiet vs. wild) — that read is a real, documented edge.",
    "Mechanism real, fortune a screenshot.",
    "And all the numbers come from ~60 days of data — about one OpEx cycle.",
    "same-day (0DTE) options are now >50% of SPX options volume",
    "plausibly earns Sharpe ~0.7–1.3 (likely 0.5–0.9 net for a retail operator)",
    "Not +371%/6mo.",
])

found   ep05-gex-dealer-hedging/README.md line 3
        Verdict: half-real. Dealer-gamma genuinely shapes the volatility regime (quiet vs. wild) — that read is a real, documented edge.
found   ep05-gex-dealer-hedging/README.md line 3
        Mechanism real, fortune a screenshot.
found   ep05-gex-dealer-hedging/README.md line 29
        And all the numbers come from ~60 days of data — about one OpEx cycle.
found   ep05-gex-dealer-hedging/README.md line 31
        same-day (0DTE) options are now >50% of SPX options volume
found   ep05-gex-dealer-hedging/README.md line 37
        plausibly earns Sharpe ~0.7–1.3 (likely 0.5–0.9 net for a retail operator)
found   ep05-gex-dealer-hedging/README.md line 37
        Not +371%/6mo.


<a id="ep06"></a>
## Ep 6. Market-Neutral Stat-Arb

**Stamp: NOT AT RETAIL.**

**The test.** A home-built market-neutral statistical arbitrage bot, first on ETF and basket pairs, then on a PCA residual strategy that bets each stock's leftover wobble snaps back. The bar was MAR (yearly return divided by the worst drop) ≥ 0.5 and Sortino (return measured against the bad days only) ≥ 1.0, net of 2× modeled cost, on an untouched holdout. The kill test deleted the five best names out of 96.

**The data.** 96 liquid US large caps that exist today, 8 years of daily data, with spreads measured from consolidated (SIP) quotes. The universe is survivorship-biased, which makes the results optimistic.

**Rerun it.** No code is committed. The research code runs on a paid consolidated-quote feed that can't be redistributed, and the holdout is spent. The write-up and two charts are here.

**The verdict.** Not at retail. The signal was real and cleared the bar on held-out data, but deleting the five best names flipped Sortino from 1.53 to −0.51. The keeper is the test, not the strategy.

**The numbers, quoted from the write-up:**

> Then deleting the five best names out of 96 flipped it from Sortino 1.53 to −0.51.
>
> All 15 → 0 cleared the bar.
>
> With measured spreads from real consolidated quotes (~6 bps), it flipped to a borderline pass: holdout Sortino 1.53, MAR 0.94, net of 2× cost.
>
> Only 55 of 96 names were net-positive. The top 5 were 27% of the positive P&L.
>
> Winners didn't persist from the training period into the holdout (rank correlation −0.12; last period's top 15 did worse than its bottom 15).
>
> The keeper is the test, not the strategy.

![Ep 6 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep06-statarb-postmortem/charts/chart_droptop5.png)

Receipts: [ep06-statarb-postmortem/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep06-statarb-postmortem) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=lMg_AvVw_6A)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [7]:
receipts("ep06-statarb-postmortem", [
    "Then deleting the five best names out of 96 flipped it from Sortino 1.53 to −0.51.",
    "All 15 → 0 cleared the bar.",
    "With measured spreads from real consolidated quotes (~6 bps), it flipped to a borderline pass: holdout Sortino 1.53, MAR 0.94, net of 2× cost.",
    "Only 55 of 96 names were net-positive. The top 5 were 27% of the positive P&L.",
    "Winners didn't persist from the training period into the holdout (rank correlation −0.12; last period's top 15 did worse than its bottom 15).",
    "The keeper is the test, not the strategy.",
])

found   ep06-statarb-postmortem/README.md line 3
        Then deleting the five best names out of 96 flipped it from Sortino 1.53 to −0.51.
found   ep06-statarb-postmortem/README.md line 20
        All 15 → 0 cleared the bar.
found   ep06-statarb-postmortem/README.md line 29
        With measured spreads from real consolidated quotes (~6 bps), it flipped to a borderline pass: holdout Sortino 1.53, MAR 0.94, net of 2× cost.
found   ep06-statarb-postmortem/README.md line 40
        Only 55 of 96 names were net-positive. The top 5 were 27% of the positive P&L.
found   ep06-statarb-postmortem/README.md line 49
        Winners didn't persist from the training period into the holdout (rank correlation −0.12; last period's top 15 did worse than its bottom 15).
found   ep06-statarb-postmortem/README.md line 3
        The keeper is the test, not the strategy.


<a id="ep07"></a>
## Ep 7. Kronos, the "GPT for candlesticks" AI

**Stamp: BEATEN BY A FORMULA.**

**The test.** Kronos is an open-source transformer trained on candlestick bars that predicts the next bar. Its volatility forecasts were raced against the bot's trailing-30-day estimator and EWMA(0.94), a one-line 1990s formula, on forward realised vol 10 trading days out.

**The data.** Real inference with Kronos-base (102M params) on CPU, across 9 assets (BTC, SOL, NVDA, AAPL, GLD, SLV, DBC, URA, USO) and 409 evaluation points on a roughly 6-weekly grid over 2021–2026. The README doesn't name the price data source.

**Rerun it.** No code is committed. The write-up and two charts are here, and the scoreboard chart uses illustrative per-asset values. The model itself is public (arXiv 2508.02739, GitHub shiyu-coder/Kronos).

**The verdict.** A real forecaster with no proven edge. As a volatility input it lost to both baselines on all 9 assets. The keeper is the one-line EWMA formula, which also beat the bot's own trailing-30-day estimator.

**The numbers, quoted from the write-up:**

> Verdict: a real forecaster, no proven edge — and as a volatility input it's ~4–5× worse than a one-line 1990s formula.
>
> | Kronos-base (102M) | 0.553 | 1.216 | 0.706 |
>
> It loses to both baselines on all 9 assets — zero wins.
>
> EWMA(0.94) beat the trailing-30-day incumbent in 36/36 cells on QLIKE (pooled)
>
> Kronos is a real forecaster that lost the one race that mattered.

![Ep 7 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep07-kronos-foundation-model/charts/chart_faceoff.png)

Receipts: [ep07-kronos-foundation-model/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep07-kronos-foundation-model) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=p1wj7qYU6dM)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [8]:
receipts("ep07-kronos-foundation-model", [
    "Verdict: a real forecaster, no proven edge — and as a volatility input it's ~4–5× worse than a one-line 1990s formula.",
    "| Kronos-base (102M) | 0.553 | 1.216 | 0.706 |",
    "It loses to both baselines on all 9 assets — zero wins.",
    "EWMA(0.94) beat the trailing-30-day incumbent in 36/36 cells on QLIKE (pooled)",
    "Kronos is a real forecaster that lost the one race that mattered.",
])

found   ep07-kronos-foundation-model/README.md line 3
        Verdict: a real forecaster, no proven edge — and as a volatility input it's ~4–5× worse than a one-line 1990s formula.
found   ep07-kronos-foundation-model/README.md line 30
        | Kronos-base (102M) | 0.553 | 1.216 | 0.706 |
found   ep07-kronos-foundation-model/README.md line 36
        It loses to both baselines on all 9 assets — zero wins.
found   ep07-kronos-foundation-model/README.md line 46
        EWMA(0.94) beat the trailing-30-day incumbent in 36/36 cells on QLIKE (pooled)
found   ep07-kronos-foundation-model/README.md line 54
        Kronos is a real forecaster that lost the one race that mattered.


<a id="ep08"></a>
## Ep 8. The Viral "VIX Cheat Sheet"

**Stamp: HALF-TRUE.**

**The test.** A viral cheat sheet says buy aggressively when the VIX (the market's fear gauge) spikes above 35 and sell when it drops under 15. The buy-the-panic leg was stress-tested on SPY, QQQ and IWM across the 2000–09 bear, the 2010–25 bull and the full cycle, with entry, exit and sizing rules the sheet never gave.

**The data.** Daily ^VIX, SPY, QQQ and IWM prices, auto-adjusted, over 2000–2025. The README names the tickers but not the data vendor.

**Rerun it.** No code is committed. The folder holds the write-up and four charts.

**The verdict.** Half-true. Waiting for the S&P to reclaim its 200-day after a VIX panic roughly halves buy-and-hold's drawdown (how far underwater you go), but it loses on return and never reaches a MAR (yearly growth divided by the worst drop) of 0.5. Selling when the VIX is calm is a coin flip, and "make millions" is false.

**The numbers, quoted from the write-up:**

> Then I ran the exact same stop through the 2010s bull: buy-and-hold made +702%, the stop made +12.9%.
>
> | Max drawdown | −28% | −33% | wins (BH −55% / −83%) |
>
> over 2010–2025 the one strategy that would have cleared the bar (MAR ≥ 0.5) was just buy-and-hold QQQ (MAR 0.53).
>
> No version of this overlay clears it (best full-cycle MAR 0.14).
>
> Verdict: half-true. "Buy the panic" is a real, durable edge — but it's a seatbelt, not an engine.

![Ep 8 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep08-vix-cheatsheet/charts/chart8_vs_index.png)

Receipts: [ep08-vix-cheatsheet/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep08-vix-cheatsheet) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=jBZSmC9jgxs)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [9]:
receipts("ep08-vix-cheatsheet", [
    "Then I ran the exact same stop through the 2010s bull: buy-and-hold made +702%, the stop made +12.9%.",
    "| Max drawdown | −28% | −33% | wins (BH −55% / −83%) |",
    "over 2010–2025 the one strategy that would have cleared the bar (MAR ≥ 0.5) was just buy-and-hold QQQ (MAR 0.53).",
    "No version of this overlay clears it (best full-cycle MAR 0.14).",
    "Verdict: half-true. \"Buy the panic\" is a real, durable edge — but it's a seatbelt, not an engine.",
])

found   ep08-vix-cheatsheet/README.md line 28
        Then I ran the exact same stop through the 2010s bull: buy-and-hold made +702%, the stop made +12.9%.
found   ep08-vix-cheatsheet/README.md line 43
        | Max drawdown | −28% | −33% | wins (BH −55% / −83%) |
found   ep08-vix-cheatsheet/README.md line 60
        over 2010–2025 the one strategy that would have cleared the bar (MAR ≥ 0.5) was just buy-and-hold QQQ (MAR 0.53).
found   ep08-vix-cheatsheet/README.md line 76
        No version of this overlay clears it (best full-cycle MAR 0.14).
found   ep08-vix-cheatsheet/README.md line 3
        Verdict: half-true. "Buy the panic" is a real, durable edge — but it's a seatbelt, not an engine.


<a id="ep09"></a>
## Ep 9. Buy QQQM Below Its 200-Day, Trail Out With a 2% Stop

**Stamp: BEATEN BY DOING NOTHING.**

**The test.** The rule buys when the Nasdaq-100 closes below its 200-day EMA, arms a 2% trailing stop once price recovers 2% above it, and sits in cash otherwise. It was coded exactly as specified and run against buy-and-hold on QQQ, the longer-history twin of QQQM.

**The data.** Alpaca's free IEX daily bars, split-adjusted, for QQQ from 2021-05 to 2026-09. That window holds the 2022 bear, the 2023-24 recovery and 2025.

**Rerun it.** Run backtest_ema_trail.py. It needs your own Alpaca market-data key in the ALPACA_KEY and ALPACA_SECRET environment variables (free IEX feed).

**The verdict.** Buy-and-hold wins, badly. The rule returned +35% against +121% for holding, and its 83% win rate doesn't change that. It fails the bar of a MAR (yearly growth divided by the worst drop) of at least 0.50 and a Sortino of at least 1.00.

**The numbers, quoted from the write-up:**

> Over a full Nasdaq-100 cycle the dip-buy-and-trail rule returned +35% while just owning the index returned +121%.
>
> | Total return | +35.0% | +121.0% |
>
> KPI bar (MAR ≥ 0.50 AND Sortino ≥ 1.00): FAILED — and it isn't close.
>
> In this window the 2022-04-06 entry did exactly that: it bought into the bear and sat 306 days at −14%
>
> Time in the market beats timing the dip.

![Ep 9 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep09-qqqm-dip-trail/charts/chart_curve.png)

Receipts: [ep09-qqqm-dip-trail/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep09-qqqm-dip-trail) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=YAmVIgHvxWU)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [10]:
receipts("ep09-qqqm-dip-trail", [
    "Over a full Nasdaq-100 cycle the dip-buy-and-trail rule returned +35% while just owning the index returned +121%.",
    "| Total return | +35.0% | +121.0% |",
    "KPI bar (MAR ≥ 0.50 AND Sortino ≥ 1.00): FAILED — and it isn't close.",
    "In this window the 2022-04-06 entry did exactly that: it bought into the bear and sat 306 days at −14%",
    "Time in the market beats timing the dip.",
])

found   ep09-qqqm-dip-trail/README.md line 3
        Over a full Nasdaq-100 cycle the dip-buy-and-trail rule returned +35% while just owning the index returned +121%.
found   ep09-qqqm-dip-trail/README.md line 21
        | Total return | +35.0% | +121.0% |
found   ep09-qqqm-dip-trail/README.md line 31
        KPI bar (MAR ≥ 0.50 AND Sortino ≥ 1.00): FAILED — and it isn't close.
found   ep09-qqqm-dip-trail/README.md line 41
        In this window the 2022-04-06 entry did exactly that: it bought into the bear and sat 306 days at −14%
found   ep09-qqqm-dip-trail/README.md line 3
        Time in the market beats timing the dip.


<a id="ep10"></a>
## Ep 10. MACD + the "Profitability Multiplier" Volume Filter

**Stamp: A THROTTLE, NOT AN EDGE.**

**The test.** The claim is that a MACD crossover becomes profitable once you only take it on days when volume is at least 1.5 times its 20-day average. The full system was coded and run on 55 liquid US large-caps, then compared with the same signal without the volume filter.

**The data.** Alpaca's free IEX daily bars, split-adjusted, for 55 liquid US large-caps from 2020-08 to 2026-09. Benchmarks are buy-and-hold SPY and an equal-weight hold of the same 55 names.

**Rerun it.** Run backtest_macd_rvol.py. It's self-contained and needs your own Alpaca market-data key in ALPACA_KEY and ALPACA_SECRET. It prints the base table, the drop-top-5 and cost tests, the sweep and the volume isolation.

**The verdict.** It earns about nothing net of cost, +3.0% against +134.3% for buy-and-hold SPY. The volume filter took profit factor from 1.07 to 1.05 and mainly shrank the drawdown (how far underwater you go), so it's a brake, not an engine.

**The numbers, quoted from the write-up:**

> | Total return | +3.0% | +134.3% | +184.5% |
>
> Profit factor was already 1.07 without the filter and dropped to 1.05 with it.
>
> Remove 5 of 55 (MS, MRK, AMD, ABBV, NVDA) and the whole system flips to −10.6% (MAR −0.12).
>
> At 2× cost it goes net-negative (−1.1%).
>
> That is a trade-count / exposure reducer — a brake — not a profitability multiplier.

![Ep 10 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep10-macd-volume/charts/chart_isolation.png)

Receipts: [ep10-macd-volume/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep10-macd-volume) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=xfH3zOK-v2w)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [11]:
receipts("ep10-macd-volume", [
    "| Total return | +3.0% | +134.3% | +184.5% |",
    "Profit factor was already 1.07 without the filter and dropped to 1.05 with it.",
    "Remove 5 of 55 (MS, MRK, AMD, ABBV, NVDA) and the whole system flips to −10.6% (MAR −0.12).",
    "At 2× cost it goes net-negative (−1.1%).",
    "That is a trade-count / exposure reducer — a brake — not a profitability multiplier.",
])

found   ep10-macd-volume/README.md line 18
        | Total return | +3.0% | +134.3% | +184.5% |
found   ep10-macd-volume/README.md line 41
        Profit factor was already 1.07 without the filter and dropped to 1.05 with it.
found   ep10-macd-volume/README.md line 47
        Remove 5 of 55 (MS, MRK, AMD, ABBV, NVDA) and the whole system flips to −10.6% (MAR −0.12).
found   ep10-macd-volume/README.md line 49
        At 2× cost it goes net-negative (−1.1%).
found   ep10-macd-volume/README.md line 41
        That is a trade-count / exposure reducer — a brake — not a profitability multiplier.


<a id="ep11"></a>
## Ep 11. Dollar-Cost Averaging vs. Lump-Sum

**Stamp: SAFER, NOT RICHER.**

**The test.** The standard advice says ease a windfall into the market instead of investing it all at once. At every rolling start month, investing the lump at once was compared with equal monthly buys over 6, 12 or 24 months, with the waiting cash earning the T-bill rate.

**The data.** Ken French's monthly US total-market return plus the 1-month T-bill, 1926-07 to 2026-07. The CSV is committed, so the run is fully offline.

**Rerun it.** Run dca_lumpsum_backtest.py. It's standard library only, reads the included CSV, and needs no credentials or downloads.

```bash
python3 dca_lumpsum_backtest.py
```

**The verdict.** Lump-sum wins the money contest, about 2 of 3 times and by about +3% on average when measured fairly. DCA wins the sleep contest. It caps how much unlucky timing can cost you at the price of some expected return.

**The numbers, quoted from the write-up:**

> | Lump-sum wins | 69.1% of all rolling windows | 73.4% |
>
> Let it earn T-bills, as anyone would, and LS's edge shrinks from +4.8% to +3.3%.
>
> | 24 months | 73.3% | +7.03% |
>
> Lump-sum's worst first-year drawdown in the whole record was −65.7% (1931); the median is −7.9%.
>
> Lump-sum wins the money contest (~2 of 3, ~+3% on average, fairly measured); DCA wins the sleep contest

![Ep 11 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep11-dca-vs-lumpsum/charts/chart_winrate.png)

Receipts: [ep11-dca-vs-lumpsum/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep11-dca-vs-lumpsum) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=DdNXfE4trE0)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [12]:
receipts("ep11-dca-vs-lumpsum", [
    "| Lump-sum wins | 69.1% of all rolling windows | 73.4% |",
    "Let it earn T-bills, as anyone would, and LS's edge shrinks from +4.8% to +3.3%.",
    "| 24 months | 73.3% | +7.03% |",
    "Lump-sum's worst first-year drawdown in the whole record was −65.7% (1931); the median is −7.9%.",
    "Lump-sum wins the money contest (~2 of 3, ~+3% on average, fairly measured); DCA wins the sleep contest",
])

found   ep11-dca-vs-lumpsum/README.md line 18
        | Lump-sum wins | 69.1% of all rolling windows | 73.4% |
found   ep11-dca-vs-lumpsum/README.md line 25
        Let it earn T-bills, as anyone would, and LS's edge shrinks from +4.8% to +3.3%.
found   ep11-dca-vs-lumpsum/README.md line 35
        | 24 months | 73.3% | +7.03% |
found   ep11-dca-vs-lumpsum/README.md line 45
        Lump-sum's worst first-year drawdown in the whole record was −65.7% (1931); the median is −7.9%.
found   ep11-dca-vs-lumpsum/README.md line 55
        Lump-sum wins the money contest (~2 of 3, ~+3% on average, fairly measured); DCA wins the sleep contest


<a id="ep12"></a>
## Ep 12. How Much Backtesting Is Enough?

**Stamp: NOT THE QUESTION.**

**The test.** The question is how many years of backtest data is enough. There's no new backtest. It's the six kill tests the earlier eleven episodes produced, each linked to the episode where it first fooled the channel.

**The data.** No new data. Each test points back to the receipts of the episode it came from.

**Rerun it.** No code is committed. The folder holds the write-up, a drop-the-top-five chart and a printable checklist image.

**The verdict.** It isn't a number of years. Enough is when a result survives six honest attempts to kill it: a MAR (yearly return divided by the deepest drawdown) of at least 0.5 and a Sortino of at least 1.0 at double costs, on data the rule never saw, and still standing after deleting the five best trades.

**The numbers, quoted from the write-up:**

> Verdict: it isn't a number of years.
>
> The gap-and-go bot: 537 trades, and 3 of them were 92% of the profit.
>
> | gap-and-go | 1.56 | −0.30 |
>
> 15 spreads looked perfect in-sample. On data they'd never seen, zero survived.
>
> My clever winner-picker landed at the 55th percentile of random.

![Ep 12 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep12-how-much-backtesting/charts/chart_droptop5.png)

Receipts: [ep12-how-much-backtesting/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep12-how-much-backtesting) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=nZnN7fESvZY)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [13]:
receipts("ep12-how-much-backtesting", [
    "Verdict: it isn't a number of years.",
    "The gap-and-go bot: 537 trades, and 3 of them were 92% of the profit.",
    "| gap-and-go | 1.56 | −0.30 |",
    "15 spreads looked perfect in-sample. On data they'd never seen, zero survived.",
    "My clever winner-picker landed at the 55th percentile of random.",
])

found   ep12-how-much-backtesting/README.md line 3
        Verdict: it isn't a number of years.
found   ep12-how-much-backtesting/README.md line 51
        The gap-and-go bot: 537 trades, and 3 of them were 92% of the profit.
found   ep12-how-much-backtesting/README.md line 45
        | gap-and-go | 1.56 | −0.30 |
found   ep12-how-much-backtesting/README.md line 58
        15 spreads looked perfect in-sample. On data they'd never seen, zero survived.
found   ep12-how-much-backtesting/README.md line 60
        My clever winner-picker landed at the 55th percentile of random.


<a id="ep13"></a>
## Ep 13. TQQQ "Just Buy 3x And Hold"

**Stamp: WORSE RIDE.**

**The test.** The belief is that buying TQQQ, the 3x leveraged QQQ, and holding forever means more money. Buy-and-hold TQQQ was ranked against QQQ, a 200-day trend filter and a 50/50 TQQQ/cash mix by MAR (yearly growth divided by the worst drop), then put through four kill tests.

**The data.** Split- and dividend-adjusted daily closes for TQQQ and QQQ from Alpaca SIP, 2016-01-04 to 2026-09-16. It leaves out the 2010-2015 bull run because the free full-history feeds weren't usable, which understates TQQQ's upside.

**Rerun it.** Run backtest_tqqq.py. The README calls it standard library only with no secrets in the file. It fetches its own prices, and the window above came from Alpaca, which needs your own keys. The curve data behind the charts is committed as curve_data.json.

**The verdict.** More money, until it isn't. TQQQ made the most money but failed the risk bar on a −81.7% drawdown that needed +445% to recover and 1,111 days underwater. Boring unleveraged QQQ beat it on every risk-adjusted measure.

**The numbers, quoted from the write-up:**

> It turned $1 into ~$32 (+3,097%) versus QQQ's ~$7 (+596%).
>
> Then it lost 81.7% peak-to-trough in 2022, stayed underwater for 1,111 days (3.0 years), and needed a +445% gain just to break even.
>
> | 3 | TQQQ buy & hold (the myth) | +3,097% | 38.2% | 0.47 | 1.17 | −81.7% | 2022 −79.1% | 1,111 d | 65.8% | fail |
>
> Actual TQQQ multiple of QQQ's total return: 5.20× — NOT 3.0×.
>
> Boring unleveraged QQQ beat it on every risk-adjusted measure.

![Ep 13 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep13-tqqq-leverage/charts/chart_drawdown.png)

Receipts: [ep13-tqqq-leverage/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep13-tqqq-leverage) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=0FlY7uXQ3wA)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [14]:
receipts("ep13-tqqq-leverage", [
    "It turned $1 into ~$32 (+3,097%) versus QQQ's ~$7 (+596%).",
    "Then it lost 81.7% peak-to-trough in 2022, stayed underwater for 1,111 days (3.0 years), and needed a +445% gain just to break even.",
    "| 3 | TQQQ buy & hold (the myth) | +3,097% | 38.2% | 0.47 | 1.17 | −81.7% | 2022 −79.1% | 1,111 d | 65.8% | fail |",
    "Actual TQQQ multiple of QQQ's total return: 5.20× — NOT 3.0×.",
    "Boring unleveraged QQQ beat it on every risk-adjusted measure.",
])

found   ep13-tqqq-leverage/README.md line 11
        It turned $1 into ~$32 (+3,097%) versus QQQ's ~$7 (+596%).
found   ep13-tqqq-leverage/README.md line 11
        Then it lost 81.7% peak-to-trough in 2022, stayed underwater for 1,111 days (3.0 years), and needed a +445% gain just to break even.
found   ep13-tqqq-leverage/README.md line 19
        | 3 | TQQQ buy & hold (the myth) | +3,097% | 38.2% | 0.47 | 1.17 | −81.7% | 2022 −79.1% | 1,111 d | 65.8% | fail |
found   ep13-tqqq-leverage/README.md line 34
        Actual TQQQ multiple of QQQ's total return: 5.20× — NOT 3.0×.
found   ep13-tqqq-leverage/README.md line 64
        Boring unleveraged QQQ beat it on every risk-adjusted measure.


<a id="ep14"></a>
## Ep 14. RSI-2 "It Calls The Bottom"

**Stamp: NOT AN EDGE.**

**The test.** The belief is that a deeply oversold 2-period RSI calls the bottom of a selloff. Connors' RSI-2 rule was coded faithfully on 55 liquid large-caps, with no stop-loss and a 10-day max hold, then put through four kill tests.

**The data.** Split-adjusted daily closes for 55 liquid US large-caps plus SPY from Alpaca's free IEX feed, 2020-08-03 to 2026-09-16. The window is mostly a bull market but includes the whole 2022 bear.

**Rerun it.** Run backtest_rsi2.py. It needs numpy, pandas and requests, plus your own Alpaca market-data keys. The curve data behind the charts is committed as curve_data.json.

**The verdict.** A high win rate isn't an edge. RSI-2 wins 66.8% of its trades and still loses to equal-weight holding of the same 55 stocks on total return, CAGR, MAR and Sortino. It lost money in 2022, the one real selloff in the sample.

**The numbers, quoted from the write-up:**

> But it returned +92% over the window while simply equal-weight holding the same 55 stocks returned +178%
>
> Re-run with those 5 names removed: MAR 0.30, Sortino 0.70 → FAIL.
>
> | 10 bps (2×, 20 bps round trip) | +66.4% | 8.7% | 0.44 | 0.96 | FAIL |
>
> | 2022 (the one real bear market) | −$7,367 | 166 |
>
> The average loss is 1.40× the average win.
>
> A HIGH WIN RATE IS NOT AN EDGE.

![Ep 14 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep14-rsi2-oversold/charts/chart_winrate.png)

Receipts: [ep14-rsi2-oversold/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep14-rsi2-oversold) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=McPRIaKvU8Y)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [15]:
receipts("ep14-rsi2-oversold", [
    "But it returned +92% over the window while simply equal-weight holding the same 55 stocks returned +178%",
    "Re-run with those 5 names removed: MAR 0.30, Sortino 0.70 → FAIL.",
    "| 10 bps (2×, 20 bps round trip) | +66.4% | 8.7% | 0.44 | 0.96 | FAIL |",
    "| 2022 (the one real bear market) | −$7,367 | 166 |",
    "The average loss is 1.40× the average win.",
    "A HIGH WIN RATE IS NOT AN EDGE.",
])

found   ep14-rsi2-oversold/README.md line 12
        But it returned +92% over the window while simply equal-weight holding the same 55 stocks returned +178%
found   ep14-rsi2-oversold/README.md line 29
        Re-run with those 5 names removed: MAR 0.30, Sortino 0.70 → FAIL.
found   ep14-rsi2-oversold/README.md line 38
        | 10 bps (2×, 20 bps round trip) | +66.4% | 8.7% | 0.44 | 0.96 | FAIL |
found   ep14-rsi2-oversold/README.md line 70
        | 2022 (the one real bear market) | −$7,367 | 166 |
found   ep14-rsi2-oversold/README.md line 84
        The average loss is 1.40× the average win.
found   ep14-rsi2-oversold/README.md line 97
        A HIGH WIN RATE IS NOT AN EDGE.


<a id="ep15"></a>
## Ep 15. The Momentum Factor, Retail Edition

**Stamp: JUST BETA.**

**The test.** Cross-sectional momentum ranks a basket of 29 liquid names by their trailing 12-1 and 6-1 month returns, buys the top third and shorts the bottom third, and rebalances every 21 trading days. It was run as the academic long/short version and as a long-only version, judged on MAR (yearly growth divided by the worst drop) and Sortino, then re-run without the 5 best months and without Bitcoin and Ethereum.

**The data.** Yahoo Finance daily bars through yfinance, adjusted for dividends and splits, from 2010-01-04 to 2026-06-29. The basket is broad indices, sector funds, commodities, bond funds, four mega-cap stocks, and BTC and ETH.

**Rerun it.** Run xsmom_backtest.py. It needs numpy, pandas and yfinance, no accounts or keys, and it writes RESULTS.md on every run.

```bash
python3 xsmom_backtest.py
```

**The verdict.** Not as a strategy a regular person should run. The long/short factor loses and can't be traded in a normal account, and the long-only version only passes by carrying a −68% drawdown and fails on every config once the two coins come out.

**The numbers, quoted from the write-up:**

> The effect is real. The version you could run is Bitcoin and Nvidia.
>
> Drop the short side and long-only "passes" at MAR 0.55–0.58, but with a −68% drawdown, and it leans on five great months.
>
> On the strongest config (6-1 long-only), zeroing the five best calendar months drops it to MAR 0.44, Sortino 0.97, back under the bar on both counts.
>
> Remove the two coins and all four versions fail.
>
> For a regular person: no, not as a strategy to run.

![Ep 15 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep15-momentum-factor/charts/chart_crypto.png)

Receipts: [ep15-momentum-factor/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep15-momentum-factor) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=eSLlOUJoHd4)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [16]:
receipts("ep15-momentum-factor", [
    "The effect is real. The version you could run is Bitcoin and Nvidia.",
    "Drop the short side and long-only \"passes\" at MAR 0.55–0.58, but with a −68% drawdown, and it leans on five great months.",
    "On the strongest config (6-1 long-only), zeroing the five best calendar months drops it to MAR 0.44, Sortino 0.97, back under the bar on both counts.",
    "Remove the two coins and all four versions fail.",
    "For a regular person: no, not as a strategy to run.",
])

found   ep15-momentum-factor/README.md line 13
        The effect is real. The version you could run is Bitcoin and Nvidia.
found   ep15-momentum-factor/README.md line 13
        Drop the short side and long-only "passes" at MAR 0.55–0.58, but with a −68% drawdown, and it leans on five great months.
found   ep15-momentum-factor/README.md line 34
        On the strongest config (6-1 long-only), zeroing the five best calendar months drops it to MAR 0.44, Sortino 0.97, back under the bar on both counts.
found   ep15-momentum-factor/README.md line 13
        Remove the two coins and all four versions fail.
found   ep15-momentum-factor/README.md line 57
        For a regular person: no, not as a strategy to run.


<a id="ep16"></a>
## Ep 16. The Golden Cross (50/200)

**Stamp: JUST A SEATBELT.**

**The test.** Hold the market when its 50-day moving average is above the 200-day, sit in T-bills when it crosses back below. It was compared with buy-and-hold over a century on MAR (yearly growth divided by the worst drop), then checked bear by bear, against other average pairs, and against a one-line price-above-200-day rule.

**The data.** Daily total returns of the whole US stock market from the Fama-French research factors in Ken French's data library, with the 1-month T-bill as the cash rate. The window runs 1926-07-01 to 2026-07-31.

**Rerun it.** Run backtest_golden_cross.py. It's standard library Python with no keys or accounts, downloads the Fama-French file on first run, and writes curve_data.json.

```bash
python3 backtest_golden_cross.py
```

**The verdict.** A seatbelt, not an engine. It made less money than buy-and-hold, cut the slow bear markets down, did nothing in the fast 2020 crash, and the plain 200-day line did the job better.

**The numbers, quoted from the write-up:**

> It loses money, it is a real seatbelt in slow crashes, and the famous version is the worst way to do it.
>
> Over 100 years the golden cross turned $1 into ~$6,149 (CAGR 9.1%) against buy-and-hold's ~$17,630 (CAGR 10.3%), because it sat in cash 28% of the time.
>
> In exchange it cut 2008 to −9.7% vs −54.6% and the worst-ever drawdown to −61.7% vs −84.1%.
>
> That single-line rule made 11.2% a year vs 9.1% with a worst drop of −33.8% vs −61.7% (MAR 0.33 vs 0.15).
>
> A seatbelt, not an engine, and not even the best seatbelt.

![Ep 16 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep16-golden-cross/charts/chart_twist.png)

Receipts: [ep16-golden-cross/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep16-golden-cross) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=ZGzCkADCeB0)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [17]:
receipts("ep16-golden-cross", [
    "It loses money, it is a real seatbelt in slow crashes, and the famous version is the worst way to do it.",
    "Over 100 years the golden cross turned $1 into ~$6,149 (CAGR 9.1%) against buy-and-hold's ~$17,630 (CAGR 10.3%), because it sat in cash 28% of the time.",
    "In exchange it cut 2008 to −9.7% vs −54.6% and the worst-ever drawdown to −61.7% vs −84.1%.",
    "That single-line rule made 11.2% a year vs 9.1% with a worst drop of −33.8% vs −61.7% (MAR 0.33 vs 0.15).",
    "A seatbelt, not an engine, and not even the best seatbelt.",
])

found   ep16-golden-cross/README.md line 14
        It loses money, it is a real seatbelt in slow crashes, and the famous version is the worst way to do it.
found   ep16-golden-cross/README.md line 14
        Over 100 years the golden cross turned $1 into ~$6,149 (CAGR 9.1%) against buy-and-hold's ~$17,630 (CAGR 10.3%), because it sat in cash 28% of the time.
found   ep16-golden-cross/README.md line 14
        In exchange it cut 2008 to −9.7% vs −54.6% and the worst-ever drawdown to −61.7% vs −84.1%.
found   ep16-golden-cross/README.md line 76
        That single-line rule made 11.2% a year vs 9.1% with a worst drop of −33.8% vs −61.7% (MAR 0.33 vs 0.15).
found   ep16-golden-cross/README.md line 84
        A seatbelt, not an engine, and not even the best seatbelt.


<a id="ep17"></a>
## Ep 17. HedgeFundie's Excellent Adventure (HFEA)

**Stamp: THE HEDGE BROKE.**

**The test.** HFEA holds 55% UPRO (3x the S&P 500) and 45% TMF (3x long Treasuries), rebalanced quarterly, on the idea that the bond leg cushions stock crashes. It was run on the real funds and on a simulated version back through the 1962–1981 rate rise, ranked on MAR (yearly growth divided by the worst drop) against the S&P and a 60/40 mix.

**The data.** The real funds come from Alpaca SIP daily total-return closes, 2016-01-04 to 2026-09-17. The simulation uses the Fama-French US market and a long-Treasury index built from FRED's 10-year yield, 1962-01-02 to 2026-07-31.

**Rerun it.** Run backtest_hfea.py. It's standard library Python, but the real-fund part needs your own free Alpaca market-data key. The FRED and Fama-French data download with no key.

```bash
python3 backtest_hfea.py
```

**The verdict.** The hedge broke. In the real record HFEA made about the same money as the S&P with about twice the drawdown, and in 2022 both legs fell together. Plain 60/40 won on risk-adjusted return in both windows.

**The numbers, quoted from the write-up:**

> Same money as the S&P, twice the crash, and the hedge broke in 2022.
>
> In the real 2016+ record HFEA turned $1 into $4.65 against plain S&P buy-and-hold's $4.46.
>
> In 2022 both legs fell together: UPRO −56.8%, TMF −72.6%, HFEA −64.1%, because the daily stock/bond correlation went from −0.34 (2016–19) to +0.08 (2022).
>
> Run the sim back through the 1962–1981 rate rise and HFEA made +68.9% in twenty years while plain stocks made +272%.
>
> THE HEDGE BROKE. HFEA is a leveraged bet on the 40-year bond bull, and the bull ended.

![Ep 17 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep17-hfea-leverage/charts/chart_2022.png)

Receipts: [ep17-hfea-leverage/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep17-hfea-leverage) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=uHvST5AaKwo)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [18]:
receipts("ep17-hfea-leverage", [
    "Same money as the S&P, twice the crash, and the hedge broke in 2022.",
    "In the real 2016+ record HFEA turned $1 into $4.65 against plain S&P buy-and-hold's $4.46.",
    "In 2022 both legs fell together: UPRO −56.8%, TMF −72.6%, HFEA −64.1%, because the daily stock/bond correlation went from −0.34 (2016–19) to +0.08 (2022).",
    "Run the sim back through the 1962–1981 rate rise and HFEA made +68.9% in twenty years while plain stocks made +272%.",
    "THE HEDGE BROKE. HFEA is a leveraged bet on the 40-year bond bull, and the bull ended.",
])

found   ep17-hfea-leverage/README.md line 17
        Same money as the S&P, twice the crash, and the hedge broke in 2022.
found   ep17-hfea-leverage/README.md line 17
        In the real 2016+ record HFEA turned $1 into $4.65 against plain S&P buy-and-hold's $4.46.
found   ep17-hfea-leverage/README.md line 17
        In 2022 both legs fell together: UPRO −56.8%, TMF −72.6%, HFEA −64.1%, because the daily stock/bond correlation went from −0.34 (2016–19) to +0.08 (2022).
found   ep17-hfea-leverage/README.md line 17
        Run the sim back through the 1962–1981 rate rise and HFEA made +68.9% in twenty years while plain stocks made +272%.
found   ep17-hfea-leverage/README.md line 141
        THE HEDGE BROKE. HFEA is a leveraged bet on the 40-year bond bull, and the bull ended.


<a id="ep18"></a>
## Ep 18. Covered-Call Income ETFs

**Stamp: YOUR OWN MONEY.**

**The test.** The covered-call income funds QYLD, XYLD, RYLD, JEPI and JEPQ were each held against their own index over identical dates, comparing total return with share price. A DIY test held the index and sold shares for the exact same income to see what was left.

**The data.** Alpaca SIP daily bars, both total return and raw share price for each fund, with QQQ or SPY as the index. Windows start between 2016-01-04 and 2022-05-04 depending on when each fund launched, and all end 2026-09-17.

**Rerun it.** Run backtest_covered_calls.py. It's standard library Python but needs your own free Alpaca market-data key.

```bash
python3 backtest_covered_calls.py
```

**The verdict.** The yield is real, the return isn't. Every fund trailed its own index, part of the payout was principal coming back, and holding the index while selling shares for the same cash left more money every time. The one real benefit is a smoother ride.

**The numbers, quoted from the write-up:**

> The yield is real. The return isn't.
>
> Flagship case: $10,000 in QYLD grew to $26,315. The same $10,000 in QQQ grew to $70,742 over the identical decade.
>
> And the "12% yield" is partly your own money: QYLD's share price fell −19.9% (from $23.12 to $18.51) across ten years of paying about 12%.
>
> QYLD's holder took $12,412 of income and had $8,006 of fund left. Someone who held QQQ and sold shares for the same $12,412 had $33,015 left.
>
> PAID WITH YOUR OWN MONEY. The yield is real, the return isn't.

![Ep 18 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep18-covered-call-etfs/charts/chart_gap.png)

Receipts: [ep18-covered-call-etfs/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep18-covered-call-etfs) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=vXzpa4JkChg)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [19]:
receipts("ep18-covered-call-etfs", [
    "The yield is real. The return isn't.",
    "Flagship case: $10,000 in QYLD grew to $26,315. The same $10,000 in QQQ grew to $70,742 over the identical decade.",
    "And the \"12% yield\" is partly your own money: QYLD's share price fell −19.9% (from $23.12 to $18.51) across ten years of paying about 12%.",
    "QYLD's holder took $12,412 of income and had $8,006 of fund left. Someone who held QQQ and sold shares for the same $12,412 had $33,015 left.",
    "PAID WITH YOUR OWN MONEY. The yield is real, the return isn't.",
])

found   ep18-covered-call-etfs/README.md line 23
        The yield is real. The return isn't.
found   ep18-covered-call-etfs/README.md line 23
        Flagship case: $10,000 in QYLD grew to $26,315. The same $10,000 in QQQ grew to $70,742 over the identical decade.
found   ep18-covered-call-etfs/README.md line 23
        And the "12% yield" is partly your own money: QYLD's share price fell −19.9% (from $23.12 to $18.51) across ten years of paying about 12%.
found   ep18-covered-call-etfs/README.md line 23
        QYLD's holder took $12,412 of income and had $8,006 of fund left. Someone who held QQQ and sold shares for the same $12,412 had $33,015 left.
found   ep18-covered-call-etfs/README.md line 110
        PAID WITH YOUR OWN MONEY. The yield is real, the return isn't.


<a id="ep19"></a>
## Ep 19. Sell in May and Go Away

**Stamp: STAY IN MAY.**

**The test.** Hold the market November through April and sit in T-bills May through October, every year, against plain buy and hold. Kill tests checked whether summer is actually negative, how much of the gap comes from a few crash years, whether the edge faded after it got famous, and what trading twice a year costs.

**The data.** Daily total returns of the US stock market from the Fama-French research factors in Ken French's data library, with the 1-month T-bill as cash. The window runs 1926-07-01 to 2026-07-31.

**Rerun it.** Run backtest_sell_in_may.py. It's standard library Python with no key needed. If the Fama-French download fails it falls back to Alpaca data, which needs your own key and only covers 2016 on.

```bash
python3 backtest_sell_in_may.py
```

**The verdict.** A real season you shouldn't trade. Summer is the weaker half but still positive, so sitting in cash every summer cost about 1.8% a year and most of a century's wealth for a modest drawdown cut.

**The numbers, quoted from the write-up:**

> The season is real. The trade is not.
>
> It averaged +3.9% and was up in 73% of years.
>
> Over the century that turned $1 into ~$3,502 against buy and hold's ~$17,630, so sitting in cash half of every year kept you just ~20% of the wealth.
>
> Removing just 3 of 101 summers erases 26.8% of the entire seasonal edge.
>
> A REAL SEASON YOU SHOULDN'T TRADE.
>
> Don't sell in May. At most, buy in November.

![Ep 19 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep19-sell-in-may/charts/chart_seasons.png)

Receipts: [ep19-sell-in-may/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep19-sell-in-may) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=IAjW0ItoAzU)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [20]:
receipts("ep19-sell-in-may", [
    "The season is real. The trade is not.",
    "It averaged +3.9% and was up in 73% of years.",
    "Over the century that turned $1 into ~$3,502 against buy and hold's ~$17,630, so sitting in cash half of every year kept you just ~20% of the wealth.",
    "Removing just 3 of 101 summers erases 26.8% of the entire seasonal edge.",
    "A REAL SEASON YOU SHOULDN'T TRADE.",
    "Don't sell in May. At most, buy in November.",
])

found   ep19-sell-in-may/README.md line 13
        The season is real. The trade is not.
found   ep19-sell-in-may/README.md line 13
        It averaged +3.9% and was up in 73% of years.
found   ep19-sell-in-may/README.md line 13
        Over the century that turned $1 into ~$3,502 against buy and hold's ~$17,630, so sitting in cash half of every year kept you just ~20% of the wealth.
found   ep19-sell-in-may/README.md line 62
        Removing just 3 of 101 summers erases 26.8% of the entire seasonal edge.
found   ep19-sell-in-may/README.md line 102
        A REAL SEASON YOU SHOULDN'T TRADE.
found   ep19-sell-in-may/README.md line 102
        Don't sell in May. At most, buy in November.


<a id="ep20"></a>
## Ep 20. 4-Hour vs Daily, Crypto: The Re-Test

**Stamp: REVERSED.**

**The test.** A long-only breakout trend follower (buy on a new high close, exit on a trailing stop or a new low close) ran on BTC and ETH with identical rules on daily and 4-hour bars, with the stop width held to the same calendar days on both. It re-tests a May 2026 research note that said the daily won, and works out why the answer flipped.

**The data.** Alpaca crypto bars for BTC/USD and ETH/USD, 2021-09-16 to 2026-09-16, with 0.10% slippage per side on every fill.

**Rerun it.** You can't fully. run_ep13.py imports private modules that aren't published, so the results are checkable only from stats_full.json and the logs in the folder.

**The verdict.** Reversed. The 4-hour beat the daily on every setting, but Ethereum did most of the work, Bitcoin leaned on one trade, the 4-hour wasn't charged for trading about 3 times as often, and it's one five-year window. The old daily-wins note was wrong because its Ethereum 4-hour download was cut short.

**The numbers, quoted from the write-up:**

> The 4-hour won, and it wasn't close.
>
> +$82,798 vs +$54,489 over five years, MAR 0.64 vs 0.32, Sortino 1.76 vs 0.92, and a shallower worst drop, 9.4% vs 12.7%.
>
> Every 4-hour setting beat every daily setting on MAR.
>
> The May 4-hour run sat on an ETH/USD 4-hour download that stopped at 2021-04-28, about 4 months of a 5-year test.
>
> Take that one trade out and the BTC 4-hour book is down about $14,560.
>
> REVERSED. Should you believe the 4-hour beats the daily? Not the way it's sold.

![Ep 20 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep20-crypto-4h-vs-daily/charts/chart_flip.png)

Receipts: [ep20-crypto-4h-vs-daily/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep20-crypto-4h-vs-daily) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=FhLEjiG4i68)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [21]:
receipts("ep20-crypto-4h-vs-daily", [
    "The 4-hour won, and it wasn't close.",
    "+$82,798 vs +$54,489 over five years, MAR 0.64 vs 0.32, Sortino 1.76 vs 0.92, and a shallower worst drop, 9.4% vs 12.7%.",
    "Every 4-hour setting beat every daily setting on MAR.",
    "The May 4-hour run sat on an ETH/USD 4-hour download that stopped at 2021-04-28, about 4 months of a 5-year test.",
    "Take that one trade out and the BTC 4-hour book is down about $14,560.",
    "REVERSED. Should you believe the 4-hour beats the daily? Not the way it's sold.",
])

found   ep20-crypto-4h-vs-daily/README.md line 20
        The 4-hour won, and it wasn't close.
found   ep20-crypto-4h-vs-daily/README.md line 20
        +$82,798 vs +$54,489 over five years, MAR 0.64 vs 0.32, Sortino 1.76 vs 0.92, and a shallower worst drop, 9.4% vs 12.7%.
found   ep20-crypto-4h-vs-daily/README.md line 46
        Every 4-hour setting beat every daily setting on MAR.
found   ep20-crypto-4h-vs-daily/README.md line 72
        The May 4-hour run sat on an ETH/USD 4-hour download that stopped at 2021-04-28, about 4 months of a 5-year test.
found   ep20-crypto-4h-vs-daily/README.md line 93
        Take that one trade out and the BTC 4-hour book is down about $14,560.
found   ep20-crypto-4h-vs-daily/README.md line 111
        REVERSED. Should you believe the 4-hour beats the daily? Not the way it's sold.


<a id="ep21"></a>
## Ep 21. Buffett's Rules, Tested

**Stamp: NOT VALIDATED.**

**The test.** A Buffett-style screen with hard quality gates, a quality score and a price score picked the top 15 S&P 500 names each year, equal weight, rebalanced annually. The kill test removed the best 5 names to see whether a few big winners carried the result.

**The data.** Fundamentals from SEC EDGAR, point-in-time by filing date, with daily closes for prices, 2015-01-02 to 2025-01-02. The universe is today's S&P 500, so it's survivorship-biased.

**Rerun it.** The code isn't committed. It shares scoring code with a private screener, so the folder holds the charts and results.json, and the results are checkable only from those numbers.

**The verdict.** Not validated, not disproven. It passes the headline bar but fails the kill test, and the survivorship-biased universe makes the results an upper bound. It's kept as a discovery tool, with no capital.

**The numbers, quoted from the write-up:**

> not validated, not disproven. Passes the headline bar, fails the kill-test, and the universe is survivorship-biased.
>
> | Screener | 21.44% | −35.15% | 0.61 | 1.31 | +595.6% |
>
> | Screener, best 5 names removed | 17.66% | −37.42% | 0.47 | 1.14 | +407.0% |
>
> Requiring 5 years of filing history removed it and cost 2.7 points of CAGR and 0.11 of MAR.
>
> Call: not validated, no capital. Kept as a discovery tool that produces a short list for human review.

![Ep 21 chart](https://raw.githubusercontent.com/backtest-graveyard/backtest-graveyard/main/ep21-buffett-screener/charts/chart_killtest.png)

Receipts: [ep21-buffett-screener/](https://github.com/backtest-graveyard/backtest-graveyard/tree/main/ep21-buffett-screener) · Episode: [watch on YouTube](https://www.youtube.com/watch?v=SajVInkvCTQ)

*Not financial advice. A backtest is a simulation of the past, not a prediction.*

In [22]:
receipts("ep21-buffett-screener", [
    "not validated, not disproven. Passes the headline bar, fails the kill-test, and the universe is survivorship-biased.",
    "| Screener | 21.44% | −35.15% | 0.61 | 1.31 | +595.6% |",
    "| Screener, best 5 names removed | 17.66% | −37.42% | 0.47 | 1.14 | +407.0% |",
    "Requiring 5 years of filing history removed it and cost 2.7 points of CAGR and 0.11 of MAR.",
    "Call: not validated, no capital. Kept as a discovery tool that produces a short list for human review.",
])

found   ep21-buffett-screener/README.md line 4
        not validated, not disproven. Passes the headline bar, fails the kill-test, and the universe is survivorship-biased.
found   ep21-buffett-screener/README.md line 26
        | Screener | 21.44% | −35.15% | 0.61 | 1.31 | +595.6% |
found   ep21-buffett-screener/README.md line 27
        | Screener, best 5 names removed | 17.66% | −37.42% | 0.47 | 1.14 | +407.0% |
found   ep21-buffett-screener/README.md line 48
        Requiring 5 years of filing history removed it and cost 2.7 points of CAGR and 0.11 of MAR.
found   ep21-buffett-screener/README.md line 76
        Call: not validated, no capital. Kept as a discovery tool that produces a short list for human review.


## The tally

One line to confirm the whole notebook against the repo.

In [23]:
print(f"{sum(TALLY)} of {len(TALLY)} quoted figures found in the committed write-ups.")
if not all(TALLY):
    print("Something above says MISSING. The write-up changed after this notebook was built.")

112 of 112 quoted figures found in the committed write-ups.


Found an error? Open an issue on the receipts repo. Corrections get pinned.

*Not financial advice. A backtest is a simulation of the past, not a prediction.*